# PROMPT ENGINEERING AND RAG
## SESSION 1

Assignment questions and answers


## Question 1

A food-delivery customer-support chatbot gives
inconsistent answers: sometimes it refuses valid refund requests and
sometimes approves them without checking order status. The cause is
vague prompts that do not specify the expected output format or decision
rules. Explain what distinguishes a clear prompt from an unclear prompt
in this context. Describe at least two specific prompt changes that
would improve consistency and reliability, and justify each change.

**Answer:**

**Difference Between Clear and Unclear Prompts**
A clear prompt explicitly defines the chatbot's role, exact decision rules (e.g., qualifying conditions for refunds), required validation steps, and a structured output format. An unclear prompt relies on broad instructions like "handle customer refunds," leaving the criteria for approval, the verification process, and the response structure open to the model's interpretation.

**Prompt Changes to Improve Consistency and Reliability**

1. **Add explicit, step-by-step decision rules and conditions**
   * *Change:* Instruct the model to check specific conditions before acting (e.g., "Only approve a refund if the order status is 'Delivered late by >45 minutes' or 'Items missing'; otherwise, reject it and explain why.").
   * *Justification:* It removes ambiguity, preventing the model from guessing whether to approve or reject a request based solely on customer tone.

2. **Enforce a strict, structured output format (e.g., JSON)**
   * *Change:* Require the model to return a fixed schema, such as:
     `{"check_status": true/false, "refund_approved": true/false, "reason": "...", "customer_message": "..."}`.
   * *Justification:* It forces the model to evaluate the required logic fields consistently and makes the response easy to validate and parse programmatically before executing actions.

## Question 2

A food-delivery complaint classifier must label messages as ‘Late
    Delivery’, ‘Wrong Item’, ‘Missing Item’, or ‘Poor Quality’. You have
    200 labelled examples, but the LLM has a limited context window and
    cannot accept all examples at once. Compare zero-shot and few-shot
    prompting, recommend an approach for this situation, and explain
    what determines how many examples to include in a few-shot prompt.

**Answer:**

### Comparison: Zero-Shot vs. Few-Shot Prompting

| Feature | Zero-Shot Prompting | Few-Shot Prompting |
| :--- | :--- | :--- |
| **Input Examples** | Provides no labelled examples, only instructions. | Provides a small set of demonstration examples (input-output pairs). |
| **Performance** | Lower accuracy; may misinterpret nuanced or overlapping classes. | Higher accuracy; guides the model on edge cases and exact output format. |
| **Context Window Usage** | Minimal token usage. | Consumes more tokens depending on the number of examples. |

---

### Recommended Approach
**Few-shot prompting** is recommended. With 200 available labelled examples, selecting a small, balanced subset will significantly improve the model's ability to distinguish between overlapping categories (like 'Wrong Item' vs. 'Missing Item') while staying well within the context window limit.

---

### Factors Determining the Number of Examples in a Few-Shot Prompt
* **Context window capacity:** The total token budget remaining after accounting for system instructions, the query, and the expected response.
* **Class representation:** The need to provide a balanced demonstration for all target classes (at least 1–2 examples per category).
* **Length of each example:** Longer customer messages reduce how many examples can fit before exceeding the token limit.
* **Diminishing returns:** Performance typically plateaus after a certain number of examples (usually 4 to 8 total), making extra examples unnecessarily costly.

## Question 3

A delivery-operations assistant must calculate the best stop
    sequence for a delivery agent using five order addresses, estimated
    traffic delays, and priority levels. Direct questions to the LLM
    produce incorrect answers without reasoning. Explain how
    Chain-of-Thought (CoT) prompting could improve the output. Give a
    brief example of a restructured step-by-step prompt and identify one
    limitation of this technique.

**Answer:**

**How Chain-of-Thought (CoT) Prompting Improves the Output**
CoT forces the LLM to break down the complex routing problem into explicit intermediate reasoning steps instead of guessing the final order immediately. This allows the model to systematically evaluate trade-offs—such as balancing high-priority deliveries against heavy traffic delays—reducing logical errors and producing an optimized, verifiable stop sequence.

**Restructured Step-by-Step Prompt Example**
```text
You are a delivery routing assistant. Plan the optimal sequence for 5 deliveries using the provided addresses, traffic delays, and priority levels. 

Follow these steps:
1. List the 5 orders and sort them by priority level.
2. For each high-priority order, evaluate the estimated travel time and traffic delays.
3. Identify which lower-priority orders can be fulfilled along the way without causing late penalties for high-priority ones.
4. Draft the route step-by-step, explaining the reason for the order of each stop.
5. Provide the final stop sequence.
```

**One Limitation**
* **Increased Token Cost and Latency:** Generating intermediate reasoning steps significantly increases the output length, leading to slower response times and higher API inference costs.

## Question 4

An internal tool lets delivery partners query a 150-page policy
    handbook covering payout rules, SLA terms, and dispute resolution.
    The handbook is updated every quarter. Justify using RAG rather than
    fine-tuning for this use case. Address at least two reasons,
    including how each approach handles document updates, and identify a
    scenario in which fine-tuning would be the better choice.

**Answer:**

**Justification for RAG over Fine-Tuning:**

1. **Handling Document Updates:** RAG updates instantly by replacing or re-indexing the modified handbook sections in the vector database with minimal computational cost. Fine-tuning requires retraining the model every quarter, which is computationally expensive, time-consuming, and risks the model confusing outdated rules with new ones.
2. **Factual Accuracy and Source Attribution:** Policy documents require precise facts regarding payouts and SLAs. RAG grounds answers directly in retrieved text passages and provides exact page or section citations, drastically reducing hallucinations. Fine-tuning stores information implicitly in weights, making it prone to hallucinating numbers and incapable of providing direct source citations.

**Scenario Where Fine-Tuning is Better:**

Fine-tuning is the better choice when the goal is to adapt the model's writing style, tone, or output structure—such as training a model to strictly adhere to a specialized JSON schema or emulate a specific brand voice—rather than teaching it new, frequently changing factual knowledge.

## Question 5

A RAG pipeline searches 500 restaurant menus stored as PDFs, each
    averaging 8 pages. Retrieved chunks sometimes cut off mid-sentence
    or separate a dish name from its price. Explain the role of chunk
    size and chunk overlap. Describe how you would adjust both
    parameters to improve retrieval quality and identify one trade-off
    introduced by your adjustments.

**Answer:**

**Role of Chunk Size and Chunk Overlap**
*   **Chunk Size:** Defines the amount of text (tokens or characters) contained in a single searchable unit. It balances providing enough context against diluting specific details.
*   **Chunk Overlap:** Sets the number of shared tokens between consecutive chunks. It preserves context across boundaries to prevent splitting related information, such as sentences or dish-and-price pairs.

**Adjustments to Improve Retrieval Quality**
*   **Increase Chunk Size:** Expand the size to fit complete menu sections or full item descriptions, ensuring dish names, descriptions, and prices remain together.
*   **Increase Chunk Overlap:** Add a moderate overlap (e.g., 15–20% of chunk size) so items falling near boundary cuts are fully captured in at least one chunk.

**Trade-off**
*   **Increased Storage and Latency:** Larger chunks with more overlap increase the total number of tokens indexed and retrieved, leading to higher vector database storage costs, higher LLM context-window costs, and slower response times.

## Question 6

A RAG complaint-resolution system stores complaint-summary
    embeddings in a FAISS index. Query speed is excellent, but results
    sometimes include outdated records for resolved complaints that
    should have been removed. A team member suggests switching to
    ChromaDB. Compare FAISS and ChromaDB, give one scenario where each
    is a better fit, explain whether switching would solve the
    outdated-record problem, and identify an additional step required
    regardless of the vector store used.

**Answer:**

| Feature | FAISS | ChromaDB |
| :--- | :--- | :--- |
| **Type** | Low-level vector similarity search library | Full-featured, embedded vector database |
| **CRUD Support** | Hard to update/delete; often requires full index rebuilds | Native support for document updates and deletions |
| **Metadata Filtering** | Limited / requires manual implementation | Built-in, rich metadata filtering |
| **Speed/Scale** | Highly optimized for massive scale and raw search speed | Optimized for smaller-to-medium scale with ease of use |

* **Scenario where FAISS is better:** A large-scale system with hundreds of millions of static vectors requiring maximum query speed and low memory overhead, where records are rarely updated or deleted.
* **Scenario where ChromaDB is better:** An interactive application with dynamic, smaller-scale datasets requiring frequent document additions, deletions, updates, and metadata-based filtering.

**Will switching solve the problem?**
Yes. Switching to ChromaDB directly solves the problem because it natively supports deleting and updating vectors by ID, whereas standard FAISS indexes make removing individual records difficult.

**Additional required step:**
Regardless of the vector store used, the application must implement an automated trigger, sync pipeline, or status check (via metadata filtering or source database updates) that flags a complaint as "resolved" and immediately initiates the removal or exclusion of that record from the search space.